In [1]:
import json
import re
from pathlib import Path
from typing import Any, Dict, List, Set, Tuple

import pandas as pd

In [2]:
EVALUATION_ROOT = Path.cwd().resolve()

RESULTS_DIR = EVALUATION_ROOT / "eval_outputs" / "final"

PREDICTION_DIR = EVALUATION_ROOT / "pipeline_outputs"

GT_PATH = EVALUATION_ROOT / "baseline" / "Ground_truth_OBD_dependency.xlsx"

GT_SHEET_NAME = "OBD_Matrix"
DIRECTED_GT = True
ROW_IS_PREREQUISITE = False
ID_WIDTH = 3

print("Evaluation root:", EVALUATION_ROOT)
print("Prediction directory:", PREDICTION_DIR)
print("Results will be saved to directory:", RESULTS_DIR)
print("Predictions will be loaded from directory:", PREDICTION_DIR)
print("Ground truth exists:", GT_PATH.exists())
print("Ground truth sheet name:", GT_SHEET_NAME)
print("Directed ground truth:", DIRECTED_GT)
print("Row is prerequisite:", ROW_IS_PREREQUISITE)
print("ID width:", ID_WIDTH)

Evaluation root: C:\Users\daoha\Downloads\IT_Tasks\BA_USDependency\eval_with_colab\dependency_analysis
Prediction directory: C:\Users\daoha\Downloads\IT_Tasks\BA_USDependency\eval_with_colab\dependency_analysis\pipeline_outputs
Results will be saved to directory: C:\Users\daoha\Downloads\IT_Tasks\BA_USDependency\eval_with_colab\dependency_analysis\eval_outputs\final
Predictions will be loaded from directory: C:\Users\daoha\Downloads\IT_Tasks\BA_USDependency\eval_with_colab\dependency_analysis\pipeline_outputs
Ground truth exists: True
Ground truth sheet name: OBD_Matrix
Directed ground truth: True
Row is prerequisite: False
ID width: 3


**LOAD GROUND TRUTH**

In [3]:
def normalize_id(raw) -> str:
    """Turn 'US_1', 'US-1', 'US-001', 1, etc. into a canonical 'US-001' form."""
    s = str(raw).strip()
    m = re.search(r"(\d+)", s)
    if not m:
        raise ValueError(f"Could not extract a story number from id: {raw!r}")
    num = int(m.group(1))
    return f"US-{num:0{ID_WIDTH}d}"

In [4]:
def _detect_header_row(path: str, sheet_name: str, corner_label: str = "US_i \\ US_j") -> int:
    """The GT sheet has a title + description row above the real header, and the
    description row happens to contain the substrings 'US_i'/'US_j' too -- so we
    require an EXACT match on the corner cell to find the true header row."""
    raw = pd.read_excel(path, sheet_name=sheet_name, header=None)
    for i, val in raw.iloc[:, 0].items():
        if isinstance(val, str) and val.strip() == corner_label:
            return i
    raise ValueError(
        f"Could not find header row with corner label {corner_label!r} in sheet {sheet_name!r}. "
        f"Inspect the sheet manually with pd.read_excel(path, sheet_name=sheet_name, header=None)."
    )

In [5]:
def load_ground_truth(path: str):
    """Read the GT matrix sheet and return:
    - gt_pairs: set of frozenset({id_i, id_j}) -- undirected, for TP/FP/FN scoring
    - gt_directed: set of (prerequisite_id, dependent_id) tuples -- for direction scoring
    - full_ids: every story ID present in the GT sheet, in original row order
    """
    header_row_idx = _detect_header_row(path, GT_SHEET_NAME)
    df = pd.read_excel(path, sheet_name=GT_SHEET_NAME, header=header_row_idx, index_col=0)

    gt_pairs, gt_directed = set(), set()
    full_ids = [normalize_id(x) for x in df.index]

    for row_label in df.index:
        for col_label in df.columns:
            val = df.loc[row_label, col_label]
            if pd.isna(val):
                continue
            val_str = str(val).strip()
            if val_str in ("1", "1.0") or val is True:
                try:
                    i = normalize_id(row_label)
                    j = normalize_id(col_label)
                except ValueError:
                    continue
                if i == j:
                    continue
                gt_pairs.add(frozenset({i, j}))
                if DIRECTED_GT:
                    if ROW_IS_PREREQUISITE:
                        gt_directed.add((i, j))
                    else:
                        gt_directed.add((j, i))

    return gt_pairs, gt_directed, full_ids

In [6]:
gt_pairs, gt_directed, full_ids = load_ground_truth(GT_PATH)

print(f"Loaded {len(full_ids)} story IDs from GT sheet")
print(f"Found {len(gt_directed)} directed pairs (prerequisite -> dependent)")

Loaded 20 story IDs from GT sheet
Found 13 directed pairs (prerequisite -> dependent)


In [7]:
print(gt_pairs)

{frozenset({'US-009', 'US-017'}), frozenset({'US-014', 'US-013'}), frozenset({'US-001', 'US-002'}), frozenset({'US-014', 'US-005'}), frozenset({'US-011', 'US-005'}), frozenset({'US-013', 'US-017'}), frozenset({'US-005', 'US-008'}), frozenset({'US-005', 'US-017'}), frozenset({'US-020', 'US-017'}), frozenset({'US-009', 'US-004'}), frozenset({'US-009', 'US-011'}), frozenset({'US-015', 'US-017'}), frozenset({'US-009', 'US-008'})}


In [8]:
print(gt_directed)

{('US-001', 'US-002'), ('US-013', 'US-017'), ('US-005', 'US-011'), ('US-005', 'US-008'), ('US-005', 'US-014'), ('US-009', 'US-017'), ('US-017', 'US-020'), ('US-013', 'US-014'), ('US-009', 'US-008'), ('US-017', 'US-015'), ('US-009', 'US-011'), ('US-004', 'US-009'), ('US-005', 'US-017')}


**LOAD SAVED RESULTS**

In [9]:
def load_saved_results(
    results_file: Path,
    meta_file: Path,
) -> tuple[list[dict], dict]:
    results = json.loads(results_file.read_text(encoding="utf-8"))
    metadata = json.loads(meta_file.read_text(encoding="utf-8"))
    return results, metadata

In [10]:
llama_ann_corr_res, llama_ann_corr_meta = load_saved_results(
    PREDICTION_DIR / "llama3.1_8b_annotated_gt_results.json",
    PREDICTION_DIR / "llama3.1_8b_annotated_gt_meta.json",
)

llama_ann_corr_time = llama_ann_corr_meta["runtime_seconds"]
llama_ann_corr_ids = {
    normalize_id(story_id)
    for story_id in llama_ann_corr_meta["kept_ids"]
}

print(f"Loaded {len(llama_ann_corr_res)} Llama annotated-GT pair results.")
print(f"Retained stories: {len(llama_ann_corr_ids)}")

Loaded 120 Llama annotated-GT pair results.
Retained stories: 16


In [11]:
assert len(llama_ann_corr_ids) == llama_ann_corr_meta["n_kept_stories"]

print(f"Loaded {len(llama_ann_corr_res)} Llama annotated-GT pair results.")
print(f"Retained stories: {len(llama_ann_corr_ids)}")
print(f"Failed pairs: {len(llama_ann_corr_meta['failed_pairs'])}")

Loaded 120 Llama annotated-GT pair results.
Retained stories: 16
Failed pairs: 0


In [12]:
qwen_ann_corr_res, qwen_ann_corr_meta = load_saved_results(
    PREDICTION_DIR / "qwen3_8b_annotated_gt_results.json",
    PREDICTION_DIR / "qwen3_8b_annotated_gt_meta.json",
)

qwen_ann_corr_time = qwen_ann_corr_meta["runtime_seconds"]
qwen_ann_corr_ids = {
    normalize_id(story_id)
    for story_id in qwen_ann_corr_meta["kept_ids"]
}

print(f"Loaded {len(qwen_ann_corr_res)} Qwen annotated-GT pair results.")
print(f"Retained stories: {len(qwen_ann_corr_ids)}")

Loaded 120 Qwen annotated-GT pair results.
Retained stories: 16


In [13]:
assert len(qwen_ann_corr_ids) == qwen_ann_corr_meta["n_kept_stories"]

print(f"Loaded {len(qwen_ann_corr_res)} Qwen annotated-GT pair results.")
print(f"Retained stories: {len(qwen_ann_corr_ids)}")
print(f"Failed pairs: {len(qwen_ann_corr_meta['failed_pairs'])}")

Loaded 120 Qwen annotated-GT pair results.
Retained stories: 16
Failed pairs: 0


In [14]:
llama_text_res, llama_text_meta = load_saved_results(
    PREDICTION_DIR / "llama3.1_8b_text_gt_results.json",
    PREDICTION_DIR / "llama3.1_8b_text_gt_meta.json",
)

llama_text_time = llama_text_meta["runtime_seconds"]
llama_text_ids = {
    normalize_id(story_id)
    for story_id in llama_text_meta["kept_ids"]
}

In [15]:
qwen_text_res, qwen_text_meta = load_saved_results(
    PREDICTION_DIR / "qwen3_8b_text_gt_results.json",
    PREDICTION_DIR / "qwen3_8b_text_gt_meta.json",
)

qwen_text_time = qwen_text_meta["runtime_seconds"]
qwen_text_ids = {
    normalize_id(story_id)
    for story_id in qwen_text_meta["kept_ids"]
}

In [16]:
qwen_ann_llm_res, qwen_ann_llm_meta = load_saved_results(
    PREDICTION_DIR / "qwen3_8b_annotated_llm_results.json",
    PREDICTION_DIR / "qwen3_8b_annotated_llm_meta.json",
)

qwen_ann_llm_time = qwen_ann_llm_meta["runtime_seconds"]
qwen_ann_llm_ids = {
    normalize_id(story_id)
    for story_id in qwen_ann_llm_meta["kept_ids"]
}

In [17]:
from typing import Set, Tuple, Dict, Any, List

In [18]:
def _compute_prf1(tp: int, fp: int, fn: int) -> Dict[str, Any]:
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
    return {
        "precision": round(precision, 4),
        "recall": round(recall, 4),
        "f1": round(f1, 4),
        "tp": tp, "fp": fp, "fn": fn
    }

In [19]:
def extract_predicted_directed(results: List[Any], use_llm_decision: bool = True) -> Set[Tuple[str, str]]:
    """Automatically detects schema type and extracts dependency with their direction tuples."""
    pred_directed: Set[Tuple[str, str]] = set()

    for item in results:
        data = item.model_dump() if hasattr(item, "model_dump") else item

        # Case A: Annotated condition schema (has final_llm or final_nlp)
        if "final_llm" in data or "final_nlp" in data:
            found_key = "dependencyFoundLLM" if use_llm_decision else "dependencyFoundNLP"
            if not data.get(found_key, True):
                continue

            key = "final_llm" if use_llm_decision else "final_nlp"
            final_block = data.get(key, {})
            dependencies = final_block.get("dependencies", [])

        # Case B: Text condition schema (has dependencyFound and dependencies list)
        elif data.get("dependencyFound", False):
            dependencies = data.get("dependencies", [])
        else:
            dependencies = []

        # Extract directed pairs from dependencies
        for dep in dependencies:
            prereq = str(dep.get("prerequisite_id")).strip()
            dependent = str(dep.get("dependent_id")).strip()
            if prereq and dependent and prereq != dependent:
                pred_directed.add((prereq, dependent))

    return pred_directed

In [20]:
llama_annotated_predicted = extract_predicted_directed(
    llama_ann_corr_res,
    use_llm_decision=True,
)

qwen_annotated_predicted = extract_predicted_directed(
    qwen_ann_corr_res,
    use_llm_decision=True,
)

print("Llama predicted dependencies:")
print(sorted(llama_annotated_predicted))

print("\nQwen predicted dependencies:")
print(sorted(qwen_annotated_predicted))

print("\nExactly same dependency set:")
print(llama_annotated_predicted == qwen_annotated_predicted)

print("\nOnly predicted by Llama:")
print(sorted(llama_annotated_predicted - qwen_annotated_predicted))

print("\nOnly predicted by Qwen:")
print(sorted(qwen_annotated_predicted - llama_annotated_predicted))

Llama predicted dependencies:
[('US-004', 'US-009'), ('US-004', 'US-011'), ('US-005', 'US-010'), ('US-005', 'US-011'), ('US-005', 'US-014'), ('US-005', 'US-017'), ('US-009', 'US-011'), ('US-009', 'US-017')]

Qwen predicted dependencies:
[('US-004', 'US-009'), ('US-004', 'US-011'), ('US-005', 'US-011'), ('US-005', 'US-014'), ('US-005', 'US-017'), ('US-009', 'US-010'), ('US-009', 'US-011'), ('US-009', 'US-017')]

Exactly same dependency set:
False

Only predicted by Llama:
[('US-005', 'US-010')]

Only predicted by Qwen:
[('US-009', 'US-010')]


In [21]:
def evaluate_pipeline_run(
    results: List[Any],
    gt_directed: Set[Tuple[str, str]],
    valid_ids: Set[str],
    use_llm_decision: bool = True
) -> Dict[str, Any]:
    """Unified evaluation function for any pipeline condition."""
    filtered_gt_dir = {(i, j) for (i, j) in gt_directed if i in valid_ids and j in valid_ids}

    pred_directed = extract_predicted_directed(results, use_llm_decision=use_llm_decision)

    tp = len(pred_directed & filtered_gt_dir)
    fp = len(pred_directed - filtered_gt_dir)
    fn = len(filtered_gt_dir - pred_directed)

    return _compute_prf1(tp, fp, fn)

In [22]:
def evaluate_pipeline_run_undirected(results, gt_pairs, valid_ids, use_llm_decision=True) -> Dict[str, Any]:
    """Direction-agnostic: was the dependency found at all, regardless of which side got called prerequisite vs. dependent."""
    filtered_gt = set()
    for pair in gt_pairs:
        i, j = tuple(pair)
        if i in valid_ids and j in valid_ids:
            filtered_gt.add(pair)

    pred_directed = extract_predicted_directed(results, use_llm_decision=use_llm_decision)
    pred_pairs = {frozenset(t) for t in pred_directed}
    tp = len(pred_pairs & filtered_gt)
    fp = len(pred_pairs - filtered_gt)
    fn = len(filtered_gt - pred_pairs)
    return _compute_prf1(tp, fp, fn)

In [23]:
def evaluate_direction_errors(results, gt_directed, valid_ids, use_llm_decision=True) -> List[Tuple[str, str]]:
    """Pairs where the dependency was correctly found but prerequisite/dependent
    got swapped -- your COLOR_MAP's 'TP (dir X)' case, made explicit."""
    filtered_gt_dir = {(i, j) for (i, j) in gt_directed if i in valid_ids and j in valid_ids}
    pred_directed = extract_predicted_directed(results, use_llm_decision=use_llm_decision)
    reversed_gt = {(j, i) for (i, j) in filtered_gt_dir}
    return sorted(pred_directed & reversed_gt)

In [24]:
llama_text_metrics = evaluate_pipeline_run(llama_text_res, gt_directed, llama_text_ids)
llama_text_undirected = evaluate_pipeline_run_undirected(llama_text_res, gt_pairs, llama_text_ids)
llama_text_dir_errors = evaluate_direction_errors(llama_text_res, gt_directed, llama_text_ids)

llama_annotated_metrics = evaluate_pipeline_run(llama_ann_corr_res, gt_directed, llama_ann_corr_ids, use_llm_decision=True)
llama_annotated_undirected = evaluate_pipeline_run_undirected(llama_ann_corr_res, gt_pairs, llama_ann_corr_ids)
llama_annotated_dir_errors = evaluate_direction_errors(llama_ann_corr_res, gt_directed, llama_ann_corr_ids)

In [25]:
qwen_text_metrics = evaluate_pipeline_run(qwen_text_res, gt_directed, qwen_text_ids)
qwen_annotated_metrics = evaluate_pipeline_run(qwen_ann_corr_res, gt_directed, qwen_ann_corr_ids, use_llm_decision=True)

qwen_text_undirected = evaluate_pipeline_run_undirected(qwen_text_res, gt_pairs, qwen_text_ids)
qwen_annotated_undirected = evaluate_pipeline_run_undirected(qwen_ann_corr_res, gt_pairs, qwen_ann_corr_ids)

qwen_text_dir_errors = evaluate_direction_errors(qwen_text_res, gt_directed, qwen_text_ids)
qwen_annotated_dir_errors = evaluate_direction_errors(qwen_ann_corr_res, gt_directed, qwen_ann_corr_ids)

In [26]:
qwen_annotated_llm_metrics = evaluate_pipeline_run(qwen_ann_llm_res, gt_directed, qwen_ann_llm_ids, use_llm_decision=True)
qwen_annotated_llm_undirected = evaluate_pipeline_run_undirected(qwen_ann_llm_res, gt_pairs, qwen_ann_llm_ids)
qwen_annotated_llm_dir_errors = evaluate_direction_errors(qwen_ann_llm_res, gt_directed, qwen_ann_llm_ids)

In [27]:
summary_data = [
    {
        "Model": "llama3.1:8b", "Condition": "Text Dependency Analysis",
        "Precision (directed)": llama_text_metrics["precision"], "Recall (directed)": llama_text_metrics["recall"],
        "F1 (directed)": llama_text_metrics["f1"], "F1 (undirected)": llama_text_undirected["f1"],
        "Direction errors": len(llama_text_dir_errors), "Time (s)": round(llama_text_time, 2),
        "TP": llama_text_metrics["tp"], "FP": llama_text_metrics["fp"], "FN": llama_text_metrics["fn"],
    },
    {
        "Model": "llama3.1:8b", "Condition": "Annotated Dependency Analysis",
        "Precision (directed)": llama_annotated_metrics["precision"], "Recall (directed)": llama_annotated_metrics["recall"],
        "F1 (directed)": llama_annotated_metrics["f1"], "F1 (undirected)": llama_annotated_undirected["f1"],
        "Direction errors": len(llama_annotated_dir_errors), "Time (s)": round(llama_ann_corr_time, 2),
        "TP": llama_annotated_metrics["tp"], "FP": llama_annotated_metrics["fp"], "FN": llama_annotated_metrics["fn"],
    },
    {
        "Model": "qwen3:8b", "Condition": "Text Dependency Analysis",
        "Precision (directed)": qwen_text_metrics["precision"], "Recall (directed)": qwen_text_metrics["recall"],
        "F1 (directed)": qwen_text_metrics["f1"], "F1 (undirected)": qwen_text_undirected["f1"],
        "Direction errors": len(qwen_text_dir_errors), "Time (s)": round(qwen_text_time, 2),
        "TP": qwen_text_metrics["tp"], "FP": qwen_text_metrics["fp"], "FN": qwen_text_metrics["fn"],
    },
    {
        "Model": "qwen3:8b", "Condition": "Annotated Dependency Analysis",
        "Precision (directed)": qwen_annotated_metrics["precision"], "Recall (directed)": qwen_annotated_metrics["recall"],
        "F1 (directed)": qwen_annotated_metrics["f1"], "F1 (undirected)": qwen_annotated_undirected["f1"],
        "Direction errors": len(qwen_annotated_dir_errors), "Time (s)": round(qwen_ann_corr_time, 2),
        "TP": qwen_annotated_metrics["tp"], "FP": qwen_annotated_metrics["fp"], "FN": qwen_annotated_metrics["fn"],
    },
]

df_summary = pd.DataFrame(summary_data)
display(df_summary)
df_summary.to_csv(RESULTS_DIR / "rq1_summary_all_models.csv", index=False)

,Model,Condition,Precision (directed),Recall (directed),F1 (directed),F1 (undirected),Direction errors,Time (s),TP,FP,FN
0,llama3.1:8b,Text Dependency Analysis,0.0882,0.75,0.1579,0.2029,2,478.09,6,62,2
1,llama3.1:8b,Annotated Dependency Analysis,0.7500,0.75,0.7500,0.7500,0,469.93,6,2,2
2,qwen3:8b,Text Dependency Analysis,0.1333,0.25,0.1739,0.1739,0,299.61,2,13,6
3,qwen3:8b,Annotated Dependency Analysis,0.7500,0.75,0.7500,0.7500,0,432.62,6,2,2


In [28]:
summary_data_2 = [
    {
        "Model": "qwen3:8b", "Condition": "Annotated GT Dependency Analysis",
        "Precision (directed)": qwen_annotated_metrics["precision"], "Recall (directed)": qwen_annotated_metrics["recall"],
        "F1 (directed)": qwen_annotated_metrics["f1"], "F1 (undirected)": qwen_annotated_undirected["f1"],
        "Direction errors": len(qwen_annotated_dir_errors), "Time (s)": round(qwen_ann_corr_time, 2),
        "TP": qwen_annotated_metrics["tp"], "FP": qwen_annotated_metrics["fp"], "FN": qwen_annotated_metrics["fn"],
    },
    {
        "Model": "qwen3:8b", "Condition": "Annotated LLM Dependency Analysis",
        "Precision (directed)": qwen_annotated_llm_metrics["precision"], "Recall (directed)": qwen_annotated_llm_metrics["recall"],
        "F1 (directed)": qwen_annotated_llm_metrics["f1"], "F1 (undirected)": qwen_annotated_llm_undirected["f1"],
        "Direction errors": len(qwen_annotated_llm_dir_errors), "Time (s)": round(qwen_ann_llm_time, 2),
        "TP": qwen_annotated_llm_metrics["tp"], "FP": qwen_annotated_llm_metrics["fp"], "FN": qwen_annotated_llm_metrics["fn"],
    }
]

df_summary_2 = pd.DataFrame(summary_data_2)
display(df_summary_2)
df_summary_2.to_csv(RESULTS_DIR / "rq2_summary_all_models.csv", index=False)

,Model,Condition,Precision (directed),Recall (directed),F1 (directed),F1 (undirected),Direction errors,Time (s),TP,FP,FN
0,qwen3:8b,Annotated GT Dependency Analysis,0.7500,0.75,0.7500,0.7500,0,432.62,6,2,2
1,qwen3:8b,Annotated LLM Dependency Analysis,0.4211,0.80,0.5517,0.5517,0,847.41,8,11,2


In [29]:
latex_table = df_summary.to_latex(index=False, float_format="%.4f",
                                    caption="RQ1: Text-based vs. CRUD-annotated dependency detection across models",
                                    label="tab:rq1_summary")
(RESULTS_DIR / "rq1_summary.tex").write_text(latex_table, encoding="utf-8")
print(latex_table)

\begin{table}
\caption{RQ1: Text-based vs. CRUD-annotated dependency detection across models}
\label{tab:rq1_summary}
\begin{tabular}{llrrrrrrrrr}
\toprule
Model & Condition & Precision (directed) & Recall (directed) & F1 (directed) & F1 (undirected) & Direction errors & Time (s) & TP & FP & FN \\
\midrule
llama3.1:8b & Text Dependency Analysis & 0.0882 & 0.7500 & 0.1579 & 0.2029 & 2 & 478.0900 & 6 & 62 & 2 \\
llama3.1:8b & Annotated Dependency Analysis & 0.7500 & 0.7500 & 0.7500 & 0.7500 & 0 & 469.9300 & 6 & 2 & 2 \\
qwen3:8b & Text Dependency Analysis & 0.1333 & 0.2500 & 0.1739 & 0.1739 & 0 & 299.6100 & 2 & 13 & 6 \\
qwen3:8b & Annotated Dependency Analysis & 0.7500 & 0.7500 & 0.7500 & 0.7500 & 0 & 432.6200 & 6 & 2 & 2 \\
\bottomrule
\end{tabular}
\end{table}

